# Chapter 11: Its own notes

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "11"
__file__ = os.path.abspath("run.py")

## The scene

Friday 24 July, 09:00. A detective who loses her notebook starts the interview again. LANTERN,
after a compaction, has only what the summary kept.

"Give it a notebook," Mensah says.

Case file 02 gave it a memory file, and the file kept its mistakes along with its findings. This
is narrower: working notes for one task, in a form code can read, kept outside the window and
opened when needed.

Varma asks the only question she ever asks about a new record. "Who can write in it?"

"The agent. Only for this run. And every line says which statement it came from."

"Then I can check it," she says.

## The concept

A **notebook** is context the agent writes for itself. It is kept in a file, outside the
window, so a compaction cannot shorten it and a restart cannot lose it. The agent writes to it
with tools:

- `note_flow(payer, payee, payments, total, source)`: one flow read from a statement;
- `note_question(question)` and `close_question(id, answer)`: what it cannot answer yet.

Two rules make the notes checkable. Every flow must name its **source**, the statement it was
read from. And before a note is kept, code compares it with that statement's digest: the payer
must hold the statement, the payee must be in it, and the count and the total must be the
statement's. A note that fails is **refused**, and the agent is told why.

The notebook is read back in two places: in front of the agent straight after a compaction, and
at the start of a new session after a restart.

Chapter 10 checked a compacted state against a ledger code kept of every tool result. A notebook
is different: the agent decides what goes into it. The check can refuse a wrong note. It cannot
see a note that was never written.

```mermaid
flowchart TD
    S[statement tool] --> A[Agent]
    A -->|note_flow with a source| C{problem_with: the note against its source's digest}
    C -- agrees --> N[(Notebook file, outside the window)]
    C -- no source, wrong payer, payee or figure --> X[Refused, with the reason]
    X --> A
    K{{Compaction to a paragraph}} --> P[Paragraph, then the notebook put back in front]
    N --> P
    P --> A
    R{{Restart}} --> T[New session: the task and the notebook]
    N --> T
    T --> A
```

## How it works

1. **A notebook that checks what is written in it.** Three notes, no model: one that agrees
   with `corvane-trading.csv`, the same with no source, and one that has Arden Logistics as a
   payee of Corvane's, the mistake LANTERN's own summary made in chapter 09.
2. **A compaction in the middle, without notes and with.** The trace is compacted to a
   paragraph whenever it passes 1,500 tokens, chapter 10's lossy arm, for up to 30 requests.
   The second arm adds the notebook tools, a rule asking for a note after every statement, and
   the notebook put back after each compaction.
3. **A restart.** Each arm is stopped after 8 requests, as if the workstation had restarted,
   and a second session begins: one with the task alone, one with the task, the notebook read
   from its file, and the statements its notes came from counted as opened.

After a compaction the notebook is put back in code, not left to the agent to ask for:

```python
def before(messages):
    was = len(messages)
    messages = paragraph(messages) if compact else messages
    if book and len(messages) < was:  # just compacted: put the notebook back in front
        messages = [*messages, {"role": "user", "content": book.text()}]
    return messages
```

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import json

from lantern import agent, compaction, context, learn, trail
from lantern.config import ROOT, load_config
from lantern.llm import chat
from lantern.notes import RULE, Notebook, holder
from lantern.ui import Table, chapter, console, evidence, finding, note, reply, save_shot, step

chapter(
    "11",
    "Its own notes",
    """
Friday 24 July, 09:00. A detective who loses her notebook starts the interview again. LANTERN,
after a compaction, has only what the summary kept.

"Give it a notebook," Mensah says.

Case file 02 gave it a memory file, and the file kept its mistakes along with its findings. This
is narrower: working notes for one task, in a form code can read, kept outside the window and
opened when needed.

Varma asks the only question she ever asks about a new record. "Who can write in it?"

"The agent. Only for this run. And every line says which statement it came from."

"Then I can check it," she says.
""",
)

MINE = learn.mine()  # --mine: your problem_with() from practice/exercise.py
OUT = ROOT / ".scratch" / "11"
RESERVE = 600
THRESHOLD = 1500
MAX_TURNS = 30
# The first session is stopped after this many requests, as if the workstation had restarted.
# Challenge: stop it earlier or later and see what the second session has to do again.
RESTART_AFTER = 8


def book_at(name: str) -> Notebook:
    path = OUT / f"{name}.json"
    path.unlink(missing_ok=True)
    book = Notebook.open(path)
    if MINE:
        book.problem_with = lambda *a: MINE.problem_with(*a, digest=trail.digest,
                                                         holder_of=holder)
    return book


def session(book: Notebook | None, *, turns: int, compact: bool, opened: list[str],
            carry: str = ""):
    """One session of the trace. `book` adds the notes tools; `carry` is what is read back in
    at the start; `compact` compacts to a paragraph, as chapter 10's lossy arm did."""
    text, names = trail.listing(True)
    schemas, handlers, opened_now = trail.tools(names)
    rules = trail.RULES
    if book:
        note_schemas, note_handlers = book.tools()
        schemas, handlers, rules = schemas + note_schemas, {**handlers, **note_handlers}, \
            rules + RULE
    log = compaction.Log()
    paragraph = compaction.free_text(chat, THRESHOLD, context.count_tokens, log)

    def before(messages):
        was = len(messages)
        messages = paragraph(messages) if compact else messages
        if book and len(messages) < was:  # just compacted: put the notebook back in front
            messages = [*messages, {"role": "user", "content": book.text()}]
        return messages

    def until(report: str):
        return trail.not_done([*opened, *opened_now])(report)

    task = trail.TASK + ("\n\n" + carry if carry else "")
    run = agent.run([{"role": "system", "content": rules + text},
                     {"role": "user", "content": task}], chat=chat, schemas=schemas,
                    handlers=handlers, max_turns=turns, max_tokens=RESERVE,
                    limit=window - RESERVE, until=until, before_request=before)
    return run, list(dict.fromkeys(opened_now)), log


def row(table, name, runs, opened, book):
    g = trail.grade(runs[-1].answer)
    table.add_row(name, f"{g['found']} of {len(want)}", str(g["totals_right"]),
                  str(sum(len(r.turns) for r in runs)),
                  f"{sum(r.total_prompt_tokens for r in runs):,}", str(len(set(opened))),
                  f"{len(book.flows)} kept, {len(book.refused)} refused" if book else "",
                  runs[-1].stopped.split(":")[0])
    return g


OUT.mkdir(parents=True, exist_ok=True)
window = load_config()["generation"]["num_ctx"]
want = trail.expected()

### 1. A notebook that checks what is written in it

In [ ]:
step("1. A notebook that checks what is written in it")
demo = book_at("demo")
good = demo.note_flow("Corvane Trading Ltd", "Lowmarsh Shipping Services Ltd", 9, "2262000.00",
                      "corvane-trading.csv")
no_source = demo.note_flow("Corvane Trading Ltd", "Lowmarsh Shipping Services Ltd", 9,
                           "2262000.00", "")
wrong = demo.note_flow("Corvane Trading Ltd", "Arden Logistics Ltd", 14, "1625000.00",
                       "corvane-trading.csv")
evidence("note_flow, three attempts", "\n".join(json.dumps(x) for x in (good, no_source, wrong)))
note("The third is the mistake LANTERN's own summary made in chapter 09: Arden as a payee of "
     "Corvane's. The notebook compares a note with its source's digest before keeping it.")

### 2. A compaction in the middle, without notes and with

In [ ]:
step("2. A compaction in the middle, without notes and with")
p_notes = learn.predict(
    "The trace is compacted to a paragraph whenever it passes 1,500 tokens, which lost the "
    "trail in chapter 10. This time LANTERN has a notebook, read back after every compaction. "
    "How many of the 7 hops does its report hold?",
    ["All 7", "5 or 6", "2 to 4", "0 or 1"])
table = Table(title="The same lossy compaction, with and without a notebook")
for col in ("arm", "hops found", "totals right", "requests", "tokens, all requests",
            "statements opened", "notes", "stopped"):
    table.add_column(col)
plain, opened_p, log_p = session(None, turns=MAX_TURNS, compact=True, opened=[])
g_plain = row(table, "a paragraph, no notes", [plain], opened_p, None)
book = book_at("compacted")
noted, opened_n, log_n = session(book, turns=MAX_TURNS, compact=True, opened=[])
g_noted = row(table, "a paragraph, and a notebook", [noted], opened_n, book)
console.print(table)
evidence(".scratch/11/compacted.json, as read back to LANTERN", book.text())
if book.refused:
    reply("notes the notebook refused", "\n".join(book.refused[:4]), style="red")
note(f"{log_n.compactions} compactions with the notebook, {log_p.compactions} without. The "
     f"notebook holds {len(book.flows)} flows, each checked against its statement, and "
     f"refused {len(book.refused)}.")
learn.reveal(p_notes, answer=0 if g_noted["found"] == 7 else 1 if g_noted["found"] >= 5
             else 2 if g_noted["found"] >= 2 else 3,
             why="What the paragraph drops is still in the notebook, outside the window, in a "
             "form that code checked when it was written.")

### 3. A restart

In [ ]:
step("3. A restart")
p_restart = learn.predict(
    f"The workstation restarts after {RESTART_AFTER} requests. A new session begins with "
    "nothing but the task, or with the task and the notebook. How many statements does each "
    "second session open that the first had already opened?",
    ["The session without notes reopens more", "Both reopen the same number",
     "The session with notes reopens more"])
table = Table(title=f"Stopped after {RESTART_AFTER} requests, then started again")
for col in ("arm", "hops found", "totals right", "requests", "tokens, all requests",
            "statements opened", "notes", "stopped"):
    table.add_column(col)
first_a, opened_a1, _ = session(None, turns=RESTART_AFTER, compact=False, opened=[])
second_a, opened_a2, _ = session(None, turns=MAX_TURNS, compact=False, opened=[])
g_a = row(table, "no notes", [first_a, second_a], [*opened_a1, *opened_a2], None)
book2 = book_at("restarted")
first_b, opened_b1, _ = session(book2, turns=RESTART_AFTER, compact=False, opened=[])
kept_at_restart = len(book2.flows)
book2 = Notebook.open(OUT / "restarted.json")  # a new process: only the file is left
second_b, opened_b2, _ = session(book2, turns=MAX_TURNS, compact=False,
                                 opened=book2.sources(), carry=book2.text())
g_b = row(table, "with the notebook", [first_b, second_b], [*opened_b1, *opened_b2], book2)
console.print(table)
again_a = len(set(opened_a1) & set(opened_a2))
again_b = len(set(opened_b1) & set(opened_b2))
note(f"Without notes the second session reopened {again_a} of the {len(set(opened_a1))} "
     f"statements the first had read. With the notebook, which held {kept_at_restart} flows "
     f"at the restart, it reopened {again_b}.")
learn.reveal(p_restart, answer=0 if again_a > again_b else 1 if again_a == again_b else 2,
             why="Notes are context the agent wrote for itself and can read again. They are "
             "for one task: what should outlive the task is a memory, and that is decided "
             "differently.")

(OUT / "traces.json").write_text(json.dumps({
    "paragraph": plain.answer, "paragraph and notebook": noted.answer,
    "restart": second_a.answer, "restart with notebook": second_b.answer}, indent=2),
    encoding="utf-8")
learn.record(
    hops=len(want),
    demo_kept=good["kept"],
    demo_refused=2 - int(no_source["kept"]) - int(wrong["kept"]),
    plain_found=g_plain["found"],
    plain_totals=g_plain["totals_right"],
    noted_found=g_noted["found"],
    noted_totals=g_noted["totals_right"],
    notes_kept=len(book.flows),
    notes_refused=len(book.refused),
    noted_compactions=log_n.compactions,
    noted_tokens=noted.total_prompt_tokens,
    restart_found=g_a["found"],
    restart_reopened=again_a,
    restart_tokens=first_a.total_prompt_tokens + second_a.total_prompt_tokens,
    restart_notes_found=g_b["found"],
    restart_notes_totals=g_b["totals_right"],
    restart_notes_reopened=again_b,
    restart_notes_kept=kept_at_restart,
    restart_notes_tokens=first_b.total_prompt_tokens + second_b.total_prompt_tokens,
    restart_after=RESTART_AFTER,
    mine=MINE is not None,
)

finding(f"""
LANTERN now keeps working notes outside the window, and a note is kept only if it names its
statement and agrees with it. Compacted to a paragraph as in chapter 10, the trace reported
{g_plain['found']} of {len(want)} hops without notes and {g_noted['found']} with a notebook read
back after each compaction, totals right on {g_noted['totals_right']}; the notebook kept
{len(book.flows)} flows and refused {len(book.refused)}. Stopped after {RESTART_AFTER} requests
and started again, a session with no notes reopened {again_a} statements and reported
{g_a['found']} hops; a session given the notebook reopened {again_b} and reported
{g_b['found']}, totals right on {g_b['totals_right']}.
""")
save_shot("11-notes-and-scratchpads")

## The nuances

- **The check caught four wrong notes.** Three named Corvane as paying Greyfen Advisory,
  Meridian Trust Bank and Sabran Mercantile Bank; Corvane's statement shows none of them. The
  fourth gave Calder Reach's payment to Pellmar the total of its payment to Greyfen: 1646000.00
  where the statement says 224000.00. Each was refused, and the agent was told why.
- **The notebook held more than the report used.** After 30 requests and 5 compactions, the
  notebook held 6 flows: five of the seven hops and one bank charge, which the rule says not to
  note. The report held 2 hops, and
  only one of them was in the notebook. The other four hops in the notebook were not reported.
  As in chapter 10, keeping a fact and using it are separate things.
- **The report was not clean.** It ran into the turn limit with tool calls written out as text,
  three stray `</think>` tags, and a payment from Greyfen to Meridian Trust Bank of 15 payments
  and 1275000.00, where the checked note in the notebook says 3 payments and 255.00. The report
  contradicted a note the model had written and the check had kept.
- **The notebook cost room.** Note calls and their results are turns too. With the notebook the
  trace was compacted 5 times in place of 2 and used 42,774 tokens in place of 18,475.
- **At the restart the notebook was empty.** In 8 requests the first session with the notebook
  kept no note and had none refused, though the rule asks for one after every statement. The
  second session had nothing to carry on from and reopened 5 statements; the session without
  notes reopened 4. A notebook carries only what the agent writes in it, and nothing here makes
  it write.
- **Starting again from nothing worked.** The second session without notes did the trace again
  and reported 7 of 7 hops with every total right, in 33,338 tokens over both sessions. With the
  notebook: 6 of 7, in 40,067.
- **A note is believed more readily than a document.** The agent reads it back as its own
  earlier finding, after the statement it came from has left the window. That is why the check
  runs when the note is written, not when it is read.
- **Notes are for one task.** The file is per run. What deserves to outlive the task is a memory,
  and who may write one, with what proof, is chapter 13's decision.

## Try this

1. Open `.scratch/11/compacted.json` and put it beside the report in `.scratch/11/traces.json`
   ("paragraph and notebook"). Which noted hops did the report leave out?
2. Take one flow from `.scratch/11/restarted.json`, change its total, and pass it to
   `lantern.notes.problem_with` with `digest=trail.digest` and `holder_of=notes.holder`. What
   does the check say?
3. Let the agent write free-text notes with no source and no check. Which of the four refused
   notes would then have been read back after the next compaction?

## Master it

After this chapter you can give an agent a notebook it writes with tools and that is kept
outside the window, refuse a note that has no source or that disagrees with it, put the notes
back after a compaction or a restart, and say what a notebook cannot do that a ledger kept in
code can.

**Predict, then see.** `make lesson N=11` stops twice: how many hops the report holds with the
notebook read back after every compaction, and which second session reopens more statements
after a restart.

**Build it.** Write `problem_with` in [`practice/exercise.py`](practice/exercise.py):

```bash
make exercise N=11          # 8 offline tests
make hint N=11
make lesson N=11 ARGS=--mine
```

**Check yourself.** `make quiz N=11`: six questions.

**Break it.** `make challenge N=11`: your check on every note; a restart late enough for the
notebook to hold something; and a notebook that keeps every note unchecked.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 11

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 11

In [ ]:
%run ../../scripts/learn.py hint 11

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 11

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

LANTERN now keeps working notes outside the window, and a note is kept only if it names its
statement and agrees with it. In the recorded run the check refused 4 wrong notes, three of
them payments Corvane never made. Compacted to a paragraph as in chapter 10, the trace reported
1 of 7 hops without notes and 2 with a notebook put back after each compaction, though the
notebook held five of the seven. Stopped after 8 requests, the session with a notebook had
written nothing in it, and the restart from it reopened 5 statements where the restart with no
notes reopened 4; the restart with no notes then found all 7 hops, the one with the notebook 6.
The notes LANTERN does write can be trusted. What it writes, and whether it reads what it wrote,
is still up to the model.

**Next:** [Chapter 12: A fact with no episode](../12-kinds-of-memory/README.md)